# Cluster Overview

This notebook queries the OCP audit SQLite datastore to present the **Cluster Overview** report — a single-row identity card per cluster covering:

- Version & Identity
- Platform & Topology
- Node Sizing
- Network Configuration
- Access Endpoints
- Update Posture

In [18]:
import os
import sys

# Point to the datastore DB before importing schema (engine binds at import time)
os.environ["OCP_AUDIT_DB"] = os.path.join(
    os.path.dirname(os.path.abspath("__file__")), "..", "datastore", "ocp_audit.db"
)
sys.path.insert(0, os.path.join(os.path.dirname(os.path.abspath("__file__")), "..", "datastore"))

import pandas as pd
from schema.database import SessionLocal, engine
from schema.models import Cluster, ClusterOverview

session = SessionLocal()
pd.set_option("display.max_colwidth", 80)
print(f"Connected to: {engine.url}")

Connected to: sqlite:////home/vagrant/git/openshift-csv-exporter/notebook/../datastore/ocp_audit.db


In [19]:
# Shared table styling — orange header band, transparent body cells
HEADER_BG = "#E07B39"   # warm orange
HEADER_FG = "#FFFFFF"
BORDER = "1px solid #D9D9D9"

_TABLE_STYLES = [
    {"selector": "thead th", "props": [
        ("background-color", HEADER_BG),
        ("color", HEADER_FG),
        ("font-weight", "600"),
        ("text-align", "left"),
        ("padding", "8px 12px"),
        ("border", BORDER),
    ]},
    {"selector": "tbody td", "props": [
        ("background-color", "transparent"),
        ("color", "inherit"),
        ("padding", "6px 12px"),
        ("border", BORDER),
    ]},
    {"selector": "tbody tr", "props": [
        ("background-color", "transparent"),
    ]},
    {"selector": "tbody tr:hover td", "props": [
        ("background-color", "rgba(224, 123, 57, 0.12)"),
    ]},
    {"selector": "table", "props": [
        ("background-color", "transparent"),
        ("border-collapse", "collapse"),
        ("font-family", "Segoe UI, Arial, sans-serif"),
        ("font-size", "13px"),
    ]},
]

def style_table(df: pd.DataFrame):
    """Return a Styler with orange headers and subtle row striping."""
    return (
        df.style
          .hide(axis="index")
          .set_table_styles(_TABLE_STYLES)
    )


## Cluster Inventory

In [20]:
df_clusters = pd.read_sql(
    session.query(
        Cluster.id,
        Cluster.cluster_name,
        Cluster.cluster_context,
        Cluster.cluster_server,
    ).statement,
    engine,
)
print(f"{len(df_clusters)} cluster(s) in dataset")
style_table(df_clusters)

3 cluster(s) in dataset


id,cluster_name,cluster_context,cluster_server
1,ocp-prod-east,admin/api-ocp-prod-east:6443,https://api.ocp-prod-east.example.com:6443
2,ocp-staging,admin/api-ocp-staging:6443,https://api.ocp-staging.example.com:6443
3,ocp-dev,admin/api-ocp-dev:6443,https://api.ocp-dev.example.com:6443


---
## Version & Identity

OCP version, Kubernetes version, cluster ID, install date, and cluster age.

In [21]:
df_version = pd.read_sql(
    session.query(
        Cluster.cluster_name,
        ClusterOverview.ocp_version,
        ClusterOverview.kubernetes_version,
        ClusterOverview.cluster_id_ocp,
        ClusterOverview.install_date,
        ClusterOverview.cluster_age_days,
    )
    .join(Cluster, ClusterOverview.cluster_id == Cluster.id)
    .statement,
    engine,
)
style_table(df_version)

cluster_name,ocp_version,kubernetes_version,cluster_id_ocp,install_date,cluster_age_days
ocp-prod-east,4.18.28,v1.31.5,8a7b6c5d-1111-2222-3333-444455556666,2024-01-15T08:30:00Z,443
ocp-staging,4.18.24,v1.31.4,9b8c7d6e-aaaa-bbbb-cccc-ddddeeeeffff,2024-03-02T12:10:00Z,397
ocp-dev,4.18.18,v1.31.3,1a2b3c4d-5e6f-7a8b-9c0d-1e2f3a4b5c6d,2024-06-20T09:45:00Z,287


---
## Platform & Topology

Infrastructure platform and control-plane / infrastructure topology.

In [22]:
df_platform = pd.read_sql(
    session.query(
        Cluster.cluster_name,
        ClusterOverview.platform,
        ClusterOverview.control_plane_topology,
        ClusterOverview.infrastructure_topology,
    )
    .join(Cluster, ClusterOverview.cluster_id == Cluster.id)
    .statement,
    engine,
)
style_table(df_platform)

cluster_name,platform,control_plane_topology,infrastructure_topology
ocp-prod-east,AWS,HighlyAvailable,HighlyAvailable
ocp-staging,AWS,HighlyAvailable,HighlyAvailable
ocp-dev,AWS,SingleReplica,SingleReplica


---
## Node Sizing

Master, worker, infra, and total node counts per cluster.

In [23]:
df_nodes = pd.read_sql(
    session.query(
        Cluster.cluster_name,
        ClusterOverview.master_count,
        ClusterOverview.worker_count,
        ClusterOverview.infra_count,
        ClusterOverview.total_node_count,
    )
    .join(Cluster, ClusterOverview.cluster_id == Cluster.id)
    .statement,
    engine,
)
style_table(df_nodes)

cluster_name,master_count,worker_count,infra_count,total_node_count
ocp-prod-east,3,6,3,12
ocp-staging,3,4,2,9
ocp-dev,1,2,0,3


---
## Network Configuration

SDN type, cluster CIDRs, and service CIDRs.

In [24]:
df_network = pd.read_sql(
    session.query(
        Cluster.cluster_name,
        ClusterOverview.network_type,
        ClusterOverview.cluster_cidrs,
        ClusterOverview.service_cidrs,
    )
    .join(Cluster, ClusterOverview.cluster_id == Cluster.id)
    .statement,
    engine,
)
style_table(df_network)

cluster_name,network_type,cluster_cidrs,service_cidrs
ocp-prod-east,OVNKubernetes,10.128.0.0/14,172.30.0.0/16
ocp-staging,OVNKubernetes,10.128.0.0/14,172.30.0.0/16
ocp-dev,OVNKubernetes,10.128.0.0/14,172.30.0.0/16


---
## Access Endpoints

Console URL, API server URL, and default ingress domain.

In [25]:
df_endpoints = pd.read_sql(
    session.query(
        Cluster.cluster_name,
        ClusterOverview.console_url,
        ClusterOverview.api_server_url,
        ClusterOverview.default_ingress_domain,
    )
    .join(Cluster, ClusterOverview.cluster_id == Cluster.id)
    .statement,
    engine,
)
style_table(df_endpoints)

cluster_name,console_url,api_server_url,default_ingress_domain
ocp-prod-east,https://console-openshift-console.apps.ocp-prod-east.example.com,https://api.ocp-prod-east.example.com:6443,apps.ocp-prod-east.example.com
ocp-staging,https://console-openshift-console.apps.ocp-staging.example.com,https://api.ocp-staging.example.com:6443,apps.ocp-staging.example.com
ocp-dev,https://console-openshift-console.apps.ocp-dev.example.com,https://api.ocp-dev.example.com:6443,apps.ocp-dev.example.com


---
## Update Posture

Current OCP version, update channel, update state, and count of available updates.

In [26]:
df_updates = pd.read_sql(
    session.query(
        Cluster.cluster_name,
        ClusterOverview.ocp_version,
        ClusterOverview.update_channel,
        ClusterOverview.update_state,
        ClusterOverview.available_updates_count,
    )
    .join(Cluster, ClusterOverview.cluster_id == Cluster.id)
    .statement,
    engine,
)
style_table(df_updates)

cluster_name,ocp_version,update_channel,update_state,available_updates_count
ocp-prod-east,4.18.28,stable-4.18,Completed,0
ocp-staging,4.18.24,stable-4.18,Completed,2
ocp-dev,4.18.18,fast-4.18,Completed,5


---
## Full Cluster Overview

All 21 fields for every cluster in a single table.

In [27]:
df_full = pd.read_sql(
    session.query(
        Cluster.cluster_name,
        ClusterOverview.ocp_version,
        ClusterOverview.kubernetes_version,
        ClusterOverview.cluster_id_ocp,
        ClusterOverview.install_date,
        ClusterOverview.cluster_age_days,
        ClusterOverview.platform,
        ClusterOverview.control_plane_topology,
        ClusterOverview.infrastructure_topology,
        ClusterOverview.master_count,
        ClusterOverview.worker_count,
        ClusterOverview.infra_count,
        ClusterOverview.total_node_count,
        ClusterOverview.network_type,
        ClusterOverview.cluster_cidrs,
        ClusterOverview.service_cidrs,
        ClusterOverview.default_ingress_domain,
        ClusterOverview.console_url,
        ClusterOverview.api_server_url,
        ClusterOverview.update_channel,
        ClusterOverview.available_updates_count,
        ClusterOverview.update_state,
    )
    .join(Cluster, ClusterOverview.cluster_id == Cluster.id)
    .statement,
    engine,
)
print(f"{len(df_full)} cluster overview(s)")
style_table(df_full)

3 cluster overview(s)


cluster_name,ocp_version,kubernetes_version,cluster_id_ocp,install_date,cluster_age_days,platform,control_plane_topology,infrastructure_topology,master_count,worker_count,infra_count,total_node_count,network_type,cluster_cidrs,service_cidrs,default_ingress_domain,console_url,api_server_url,update_channel,available_updates_count,update_state
ocp-prod-east,4.18.28,v1.31.5,8a7b6c5d-1111-2222-3333-444455556666,2024-01-15T08:30:00Z,443,AWS,HighlyAvailable,HighlyAvailable,3,6,3,12,OVNKubernetes,10.128.0.0/14,172.30.0.0/16,apps.ocp-prod-east.example.com,https://console-openshift-console.apps.ocp-prod-east.example.com,https://api.ocp-prod-east.example.com:6443,stable-4.18,0,Completed
ocp-staging,4.18.24,v1.31.4,9b8c7d6e-aaaa-bbbb-cccc-ddddeeeeffff,2024-03-02T12:10:00Z,397,AWS,HighlyAvailable,HighlyAvailable,3,4,2,9,OVNKubernetes,10.128.0.0/14,172.30.0.0/16,apps.ocp-staging.example.com,https://console-openshift-console.apps.ocp-staging.example.com,https://api.ocp-staging.example.com:6443,stable-4.18,2,Completed
ocp-dev,4.18.18,v1.31.3,1a2b3c4d-5e6f-7a8b-9c0d-1e2f3a4b5c6d,2024-06-20T09:45:00Z,287,AWS,SingleReplica,SingleReplica,1,2,0,3,OVNKubernetes,10.128.0.0/14,172.30.0.0/16,apps.ocp-dev.example.com,https://console-openshift-console.apps.ocp-dev.example.com,https://api.ocp-dev.example.com:6443,fast-4.18,5,Completed


In [28]:
session.close()
print("Session closed.")

Session closed.
